# Vault Secrets Operator con Secretos Dinámicos

Este notebook demuestra cómo configurar y utilizar el **Vault Secrets Operator (VSO)** en Kubernetes (Minikube) para sincronizar secretos dinámicos desde HashiCorp Vault.

Cubriremos los siguientes pasos:
1. Preparar el entorno de Minikube.
2. Configurar la autenticación OIDC en Kubernetes.
3. Configurar el método de autenticación JWT en Vault.
4. Instalar VSO.
5. Crear recursos `VaultConnection` y `VaultAuth`.
6. Sincronizar secretos estáticos y certificados

Eliminamos el perfil webinar

In [114]:
! minikube delete --all

🔥  Deleting "workshop" in podman ...
🔥  Removing /Users/jose/.minikube/machines/workshop ...
💀  Removed all traces of the "workshop" cluster.
🔥  Successfully deleted all profiles


In [115]:
! minikube start -p workshop --force

😄  [workshop] minikube v1.35.0 on Darwin 26.5.1 (arm64)
❗  minikube skips various validations when --force is supplied; this may lead to unexpected behavior
✨  Using the podman (experimental) driver based on user configuration
📌  Using Podman driver with root privileges
👍  Starting "workshop" primary control-plane node in "workshop" cluster
🚜  Pulling base image v0.0.46 ...
E0616 17:36:17.259367   69865 cache.go:222] Error downloading kic artifacts:  not yet implemented, see issue #8426
🔥  Creating podman container (CPUs=2, Memory=3568MB) ...
🐳  Preparing Kubernetes v1.32.0 on Docker 27.4.1 ...E0616 17:36:39.125515   69865 start.go:132] Unable to get host IP: RoutableHostIPFromInside is currently only implemented for linux

    ▪ Generating certificates and keys ...
    ▪ Booting up control plane ...
    ▪ Configuring RBAC rules ...
🔗  Configuring bridge CNI (Container Networking Interface) ...
🔎  Verifying Kubernetes components...
    ▪ Using image gcr.io/k8s-minikube/storage-provisio

In [116]:
%%bash
helm repo add hashicorp https://helm.releases.hashicorp.com
helm repo update

"hashicorp" already exists with the same configuration, skipping
Hang tight while we grab the latest from your chart repositories...
...Successfully got an update from the "secrets-store-csi-driver" chart repository
...Successfully got an update from the "postgres-operator-charts" chart repository
...Successfully got an update from the "ingress-nginx" chart repository
...Successfully got an update from the "jaegertracing" chart repository
...Successfully got an update from the "minio" chart repository
...Successfully got an update from the "jetstack" chart repository
...Successfully got an update from the "datadog" chart repository
...Successfully got an update from the "signoz" chart repository
...Successfully got an update from the "external-secrets" chart repository
...Successfully got an update from the "jaeger-all-in-one" chart repository
...Successfully got an update from the "vault" chart repository
...Successfully got an update from the "hashicorp" chart repository
...Successfu

In [117]:
%env WORKDIR=/tmp

env: WORKDIR=/tmp


Definimos una serie de variable de entorno con las que trabajaremos a continuación

In [118]:
import os
import subprocess

# Obtain Vault address and token from Terraform output and store them in environment variables.
# This notebook lives in 9_K8s_integration_example, so the cluster state is one folder up.
cluster_state = "../1_Create_HCP_Vault_Cluster/terraform.tfstate"

terraform_output = subprocess.check_output(
    ["terraform", "output", f"-state={cluster_state}", "-raw", "hcp_vault_cluster_url"],
    text=True,
).strip()
terraform_namespace = "admin"
terraform_token = subprocess.check_output(
    ["terraform", "output", f"-state={cluster_state}", "-raw", "admin_token"],
    text=True,
).strip()

os.environ["VAULT_TOKEN"] = terraform_token
os.environ["VAULT_ADDR"] = terraform_output
os.environ["VAULT_NAMESPACE"] = terraform_namespace

print(f"VAULT_ADDR={os.environ['VAULT_ADDR']}")
print(f"VAULT_NAMESPACE={os.environ['VAULT_NAMESPACE']}")

VAULT_ADDR=https://vault-demo-cluster-public-vault-ab3160e9.c84ec63b.z1.hashicorp.cloud:8200
VAULT_NAMESPACE=admin


Creamos un directorio temporal

In [119]:
%%bash
rm -rf /tmp/vault
mkdir /tmp/vault

### Configuración de OIDC en Kubernetes

Para que Vault pueda autenticar las cargas de trabajo de Kubernetes, necesitamos configurar el clúster para emitir tokens OIDC válidos y exponer las claves públicas para su verificación.

Creamos un `ClusterRoleBinding` para permitir que el emisor de cuentas de servicio sea descubierto.

In [120]:
%%bash
kubectl create clusterrolebinding oidc-reviewer  \
   --clusterrole=system:service-account-issuer-discovery \
   --group=system:unauthenticated


clusterrolebinding.rbac.authorization.k8s.io/oidc-reviewer created


In [121]:
%%bash
ISSUER="$(kubectl get --raw /.well-known/openid-configuration | jq -r '.issuer')"
echo $ISSUER

https://kubernetes.default.svc.cluster.local


In [122]:
%%bash
kubectl get --raw "$(kubectl get --raw /.well-known/openid-configuration | jq -r '.jwks_uri' | sed -r 's/.*\.[^/]+(.*)/\1/')" | jq -r

{
  "keys": [
    {
      "use": "sig",
      "kty": "RSA",
      "kid": "VQMJ8El7PHH51M8Clf5uWJMj6POhRSgXMm57HRv78KY",
      "alg": "RS256",
      "n": "0sbCTOnrgi1DnT_qVJHszD9T1ev0V5myQtegc0cgYCKsTa8H5YGV3_OAVX28lA-GUhkSQTwVfKr-8qrN-eLgMKtW7D8OjEVIpdsV7dgXWa_FFY6jMCVItO8mafuz4oPlhHNszwuPO6JB8RiLmHQZX7sefzdJ56y0jtlc6kXFBTRedlCcWbKkw-FkDL3zPU3V65-yxARceyBViNy4VT2vulBQReU30w4mcmUHap-g2DjmWjI1HTHVwjkf7lsGp6pnlQtRi2eAqx3GURVZwyGzWsUuG3Ru8iXi1pH7Xci3VbRTTb9MYWlGRY_JPG-Yef2Bj7r-bg7c93ZpU1imEpDB7w",
      "e": "AQAB"
    }
  ]
}


In [123]:
%%bash
pip3 install pycryptodome


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip


### Generación de claves RSA para validación JWT

El siguiente script de Python extrae la clave pública del JWKS (JSON Web Key Set) de Kubernetes y la convierte a formato PEM. Esta clave pública se utilizará para configurar el validador JWT en Vault.

In [124]:
import base64
import binascii
from Crypto.PublicKey import RSA

jwks_key =  {
  "keys": [
    {
      "use": "sig",
      "kty": "RSA",
      "kid": "VQMJ8El7PHH51M8Clf5uWJMj6POhRSgXMm57HRv78KY",
      "alg": "RS256",
      "n": "0sbCTOnrgi1DnT_qVJHszD9T1ev0V5myQtegc0cgYCKsTa8H5YGV3_OAVX28lA-GUhkSQTwVfKr-8qrN-eLgMKtW7D8OjEVIpdsV7dgXWa_FFY6jMCVItO8mafuz4oPlhHNszwuPO6JB8RiLmHQZX7sefzdJ56y0jtlc6kXFBTRedlCcWbKkw-FkDL3zPU3V65-yxARceyBViNy4VT2vulBQReU30w4mcmUHap-g2DjmWjI1HTHVwjkf7lsGp6pnlQtRi2eAqx3GURVZwyGzWsUuG3Ru8iXi1pH7Xci3VbRTTb9MYWlGRY_JPG-Yef2Bj7r-bg7c93ZpU1imEpDB7w",
      "e": "AQAB"
    }
  ]
}

# Select the specific key from the JWKS
selected_key = jwks_key['keys'][0]

# Decode the base64 values of 'n' and 'e' to bytes
n_bytes = base64.urlsafe_b64decode(selected_key['n'] + '==')
e_bytes = base64.urlsafe_b64decode(selected_key['e'] + '==')

# Construct an RSA key
rsa_key = RSA.construct((int(binascii.hexlify(n_bytes), 16), int(binascii.hexlify(e_bytes), 16)))

# Convert the RSA key to PEM format
pem_key = rsa_key.exportKey(format='PEM')

print(pem_key.decode('utf-8'))


-----BEGIN PUBLIC KEY-----
MIIBIjANBgkqhkiG9w0BAQEFAAOCAQ8AMIIBCgKCAQEA0sbCTOnrgi1DnT/qVJHs
zD9T1ev0V5myQtegc0cgYCKsTa8H5YGV3/OAVX28lA+GUhkSQTwVfKr+8qrN+eLg
MKtW7D8OjEVIpdsV7dgXWa/FFY6jMCVItO8mafuz4oPlhHNszwuPO6JB8RiLmHQZ
X7sefzdJ56y0jtlc6kXFBTRedlCcWbKkw+FkDL3zPU3V65+yxARceyBViNy4VT2v
ulBQReU30w4mcmUHap+g2DjmWjI1HTHVwjkf7lsGp6pnlQtRi2eAqx3GURVZwyGz
WsUuG3Ru8iXi1pH7Xci3VbRTTb9MYWlGRY/JPG+Yef2Bj7r+bg7c93ZpU1imEpDB
7wIDAQAB
-----END PUBLIC KEY-----


### Habilitar y configurar el método de autenticación JWT en Vault

Habilitamos el método de autenticación `jwt` en Vault y lo configuramos con la clave pública que obtuvimos anteriormente. Esto permite a Vault verificar los tokens de cuenta de servicio (SA) firmados por Kubernetes.

También creamos un rol (`my-role`) que vincula la cuenta de servicio de Kubernetes (`default:default`) con una política de Vault (`devk8s`).

In [125]:
%%bash
vault auth enable -path=jwt-k8s jwt || true

Success! Enabled jwt auth method at: jwt-k8s/


In [156]:
%%bash
kubectl get --raw /openid/v1/jwks > ${WORKDIR}/kubernetes-jwks.json

python3 - <<'PY'
import base64
import binascii
import json
import os
from Crypto.PublicKey import RSA

with open(os.path.join(os.environ["WORKDIR"], "kubernetes-jwks.json"), "r", encoding="utf-8") as jwks_file:
    jwks = json.load(jwks_file)

selected_key = next(key for key in jwks["keys"] if key.get("kty") == "RSA")
n_bytes = base64.urlsafe_b64decode(selected_key["n"] + "=" * (-len(selected_key["n"]) % 4))
e_bytes = base64.urlsafe_b64decode(selected_key["e"] + "=" * (-len(selected_key["e"]) % 4))
rsa_key = RSA.construct((int(binascii.hexlify(n_bytes), 16), int(binascii.hexlify(e_bytes), 16)))

with open(os.path.join(os.environ["WORKDIR"], "kubernetes-jwt-pubkey.pem"), "wb") as pem_file:
    pem_file.write(rsa_key.exportKey(format="PEM"))
PY

vault write auth/jwt-k8s/config \
  bound_issuer="https://kubernetes.default.svc.cluster.local" \
  jwt_validation_pubkeys="$(cat ${WORKDIR}/kubernetes-jwt-pubkey.pem)"

Success! Data written to: auth/jwt-k8s/config


In [157]:
%%bash
cat > ${WORKDIR}/jwt-role.json <<EOF
{
  "role_type": "jwt",
  "bound_audiences": ["https://kubernetes.default.svc.cluster.local"],
  "user_claim": "sub",
  "bound_claims_type": "glob",
  "bound_claims": {
    "sub": "system:serviceaccount:*:*"
  },
  "policies": ["devk8s"],
  "ttl": "1h"
}
EOF

vault delete auth/jwt-k8s/role/my-role || true
vault write auth/jwt-k8s/role/my-role - < ${WORKDIR}/jwt-role.json

Success! Data deleted (if it existed) at: auth/jwt-k8s/role/my-role
Success! Data written to: auth/jwt-k8s/role/my-role


### Crear política en Vault para certificados, secretos estáticos e instant updates

Definimos una política llamada `devk8s` para el rol JWT usado por VSO. La política permite leer y suscribirse a los secretos estáticos del mount KV v2 `secrets`, consumir el event subsystem de Vault desde `sys/events/subscribe/*`, emitir certificados desde el rol PKI `jose-merchan-sbx-hashidemos-io` y revocar certificados en `pki_int/revoke`.

La parte de eventos sigue la estructura recomendada para VSO instant updates: `subscribe` y `subscribe_event_types` sobre el mount observado, más `read` sobre `sys/events/subscribe/*`.

In [158]:
%%bash
vault policy write devk8s - <<EOF
path "secrets/*" {
  capabilities = ["read", "list", "subscribe"]
  subscribe_event_types = ["*"]                     # https://developer.hashicorp.com/vault/docs/concepts/events#event-types
}
path "sys/events/subscribe/*" {
  capabilities = ["read"]
}
path "pki_int/issue/jose-merchan-sbx-hashidemos-io" {
  capabilities = [ "update" ]
}
path "pki_int/revoke" {
  capabilities = [ "update" ]
}

EOF

Success! Uploaded policy: devk8s


### Instalación del Vault Secrets Operator (VSO)

Instalamos el Vault Secrets Operator utilizando Helm. Este operador se encargará de gestionar la sincronización de secretos entre Vault y Kubernetes.

In [130]:
%%bash
helm repo add hashicorp https://helm.releases.hashicorp.com
helm repo update
# Los recursos se instalaran en el namespace vault-secrets-operator
helm install --version 1.0.1 --create-namespace --namespace vault-secrets-operator vault-secrets-operator hashicorp/vault-secrets-operator

"hashicorp" already exists with the same configuration, skipping
Hang tight while we grab the latest from your chart repositories...
...Successfully got an update from the "postgres-operator-charts" chart repository
...Successfully got an update from the "secrets-store-csi-driver" chart repository
...Successfully got an update from the "minio" chart repository
...Successfully got an update from the "jaegertracing" chart repository
...Successfully got an update from the "ingress-nginx" chart repository
...Successfully got an update from the "signoz" chart repository
...Successfully got an update from the "jetstack" chart repository
...Successfully got an update from the "datadog" chart repository
...Successfully got an update from the "jaeger-all-in-one" chart repository
...Successfully got an update from the "vault" chart repository
...Successfully got an update from the "hashicorp" chart repository
...Successfully got an update from the "external-secrets" chart repository
...Successfu

In [131]:
! kubectl get pods -n vault-secrets-operator

NAME                                                         READY   STATUS    RESTARTS   AGE
vault-secrets-operator-controller-manager-6c8df4c9b4-jf5xq   2/2     Running   0          51s


### Configurar la conexión de VSO con Vault (VaultConnection)

Creamos un recurso `VaultConnection` que define cómo el operador debe conectarse a la instancia de Vault. En este caso, apuntamos a la dirección de Vault configurada en las variables de entorno.

In [163]:
%%bash

cat > ${WORKDIR}/vso_crd.yaml <<EOF
---
apiVersion: secrets.hashicorp.com/v1beta1
kind: VaultConnection
metadata:
  namespace: vault-secrets-operator
  name: example
spec:
  address: ${VAULT_ADDR}

EOF

kubectl apply -f ${WORKDIR}/vso_crd.yaml

vaultconnection.secrets.hashicorp.com/example unchanged


In [ ]:
! kubectl describe VaultConnection example -n vault-secrets-operator

Name:         example
Namespace:    vault-secrets-operator
Labels:       <none>
Annotations:  <none>
API Version:  secrets.hashicorp.com/v1beta1
Kind:         VaultConnection
Metadata:
  Creation Timestamp:  2026-06-16T15:43:28Z
  Finalizers:
    vaultconnection.secrets.hashicorp.com/finalizer
  Generation:        1
  Resource Version:  820
  UID:               f5e42546-e31d-46a0-adcb-39aa33b152c0
Spec:
  Address:          https://vault-demo-cluster-public-vault-ab3160e9.c84ec63b.z1.hashicorp.cloud:8200
  Skip TLS Verify:  false
Status:
  Conditions:
    Last Transition Time:  2026-06-16T15:43:28Z
    Message:               Vault ping, address=https://vault-demo-cluster-public-vault-ab3160e9.c84ec63b.z1.hashicorp.cloud:8200
    Observed Generation:   1
    Reason:                Accepted
    Status:                True
    Type:                  VaultPing
    Last Transition Time:  2026-06-16T15:43:28Z
    Message:               Successfully validated resource, address=https://vault-de

### Configurar la autenticación de VSO (VaultAuth)

Creamos un recurso `VaultAuth` que especifica el método de autenticación que VSO utilizará para autenticarse con Vault. Usamos el método `jwt` y el rol `my-role` que configuramos anteriormente. Esto permite que VSO utilice la cuenta de servicio de Kubernetes para obtener un token de Vault.

In [ ]:
%%bash
cat > ${WORKDIR}/vaultauth_crd.yaml <<EOF
---
apiVersion: secrets.hashicorp.com/v1beta1
kind: VaultAuth
metadata:
  namespace: vault-secrets-operator
  name: example
spec:
  vaultConnectionRef: example
  namespace: admin
  allowedNamespaces: ["*"]
  method: jwt
  mount: jwt-k8s

  jwt:
    role: my-role
    serviceAccount: default
    audiences: ["https://kubernetes.default.svc.cluster.local"]

EOF
kubectl apply -f ${WORKDIR}/vaultauth_crd.yaml


vaultauth.secrets.hashicorp.com/example patched (no change)
--- Verificando spec resultante ---
admin

In [180]:
! kubectl describe VaultAuth example -n vault-secrets-operator

Name:         example
Namespace:    vault-secrets-operator
Labels:       <none>
Annotations:  <none>
API Version:  secrets.hashicorp.com/v1beta1
Kind:         VaultAuth
Metadata:
  Creation Timestamp:  2026-06-16T15:43:47Z
  Finalizers:
    vaultauth.secrets.hashicorp.com/finalizer
  Generation:        2
  Resource Version:  2205
  UID:               fac1651a-6ede-45cf-8ef0-6aadf2686b91
Spec:
  Allowed Namespaces:
    *
  Jwt:
    Audiences:
      https://kubernetes.default.svc.cluster.local
    Role:                      my-role
    Service Account:           default
    Token Expiration Seconds:  600
  Method:                      jwt
  Mount:                       jwt-k8s
  Namespace:                   admin
  Vault Connection Ref:        example
Status:
  Conditions:
    Last Transition Time:  2026-06-16T16:00:40Z
    Message:               VaultAuthHealthy
    Observed Generation:   2
    Reason:                Healthy
    Status:                True
    Type:                  Hea

### Crear certificado TLS con VaultPKISecret

Creamos un recurso `VaultPKISecret` para que VSO emita un certificado desde el PKI engine de Vault y lo escriba como secreto Kubernetes de tipo `kubernetes.io/tls` en el namespace `app-1`. El manifiesto usa la estructura del CRD `VaultPKISecret`: `mount`, `role`, `commonName`, `ttl`, `expiryOffset`, `destination` y `vaultAuthRef`.

In [165]:
! kubectl create namespace app-1 --dry-run=client -o yaml | kubectl apply -f -

namespace/app-1 configured


In [169]:
%%bash
cat > ${WORKDIR}/pki_secret.yaml <<EOF
apiVersion: secrets.hashicorp.com/v1beta1
kind: VaultPKISecret
metadata:
  name: vault-pki-app
  namespace: app-1
spec:
  namespace: ${VAULT_NAMESPACE:-admin}
  vaultAuthRef: vault-secrets-operator/example
  mount: pki_int
  role: jose-merchan-sbx-hashidemos-io
  destination:
    name: secretpki
    create: true
  commonName: app-1.jose-merchan.sbx.hashidemos.io
  format: pem
  revoke: true
  expiryOffset: 4h
  ttl: 14d
EOF


kubectl apply -f ${WORKDIR}/pki_secret.yaml

vaultpkisecret.secrets.hashicorp.com/vault-pki-app created


In [181]:
! kubectl describe VaultPKISecret vault-pki-app -n app-1

Name:         vault-pki-app
Namespace:    app-1
Labels:       <none>
Annotations:  <none>
API Version:  secrets.hashicorp.com/v1beta1
Kind:         VaultPKISecret
Metadata:
  Creation Timestamp:  2026-06-16T16:00:55Z
  Finalizers:
    vaultpkisecrets.secrets.hashicorp.com/finalizer
  Generation:        2
  Resource Version:  2244
  UID:               c60a51ca-901b-407c-8ea0-4b5617812380
Spec:
  Common Name:  app-1.jose-merchan.sbx.hashidemos.io
  Destination:
    Create:     true
    Name:       secretpki
    Overwrite:  false
    Transformation:
  Expiry Offset:   4h
  Format:          pem
  Mount:           pki_int
  Namespace:       admin
  Revoke:          true
  Role:            jose-merchan-sbx-hashidemos-io
  Ttl:             14d
  Vault Auth Ref:  vault-secrets-operator/example
Status:
  Conditions:
    Last Transition Time:  2026-06-16T16:00:56Z
    Message:               Secret synced, horizon=333h22m57.34380825s
    Observed Generation:   2
    Reason:                Synced


In [183]:
%%bash
kubectl get secret secretpki -n app-1 -o yaml

apiVersion: v1
data:
  _raw: eyJhdXRob3JpdHlfa2V5X2lkIjoiNzg6YTk6NjY6Zjg6Y2I6ZjM6MDE6ZGM6NjA6YjM6OGM6NDQ6YmM6Y2E6MDM6MTc6YWQ6YzM6OTA6N2YiLCJjYV9jaGFpbiI6WyItLS0tLUJFR0lOIENFUlRJRklDQVRFLS0tLS1cbk1JSUZDakNDQS9LZ0F3SUJBZ0lVSHJhN2pDMCs0eDVVblZDMVowSWx3MGF3MGswd0RRWUpLb1pJaHZjTkFRRUxcbkJRQXdLVEVuTUNVR0ExVUVBeE1lYW05elpTMXRaWEpqYUdGdUxuTmllQzVvWVhOb2FXUmxiVzl6TG1sdk1CNFhcbkRUSTJNRFl4TmpFME16VXdPRm9YRFRJMk1USXhNakU0TXpVek9Gb3dHekVaTUJjR0ExVUVBd3dRYm1WM1gybHVcbmRHVnliV1ZrYVdGMFpUQ0NBU0l3RFFZSktvWklodmNOQVFFQkJRQURnZ0VQQURDQ0FRb0NnZ0VCQUxlNlFDaXBcbitDN2JrVkVVVWY1ZmhJUzJKenJTZXRZQ1p3WDhITjdlTmU5ZWxNZHNSUlIxK2Qxb29CWjNIZlkzRWdsUk5hZmRcbkNiMFNGYlEyWTAxYW5JVzhNOVpWYS9ZV2g3TDAvS0l0cGlGd3owR3ppcS95RGc1SVI1WHcvaEw2WXc3MjU4dkRcbi9QcXlDMVpHdUhBMndtbS8vMVJMcGNrT0lyWUhNb1ozRExjWEdCeE9RNTFOSkYzVXlEODZzZ3ZUbFZkeUhQaU1cbno0d2sxQTUrTzdEdk1QOVMxVnVtZDlzckdJQlROOWwxZ2dNbzRnRjA3cllBQjRXY0JXZEpSdVAzU0MvYjF6YmtcbmswTUtaeGhvUVhTdzdKcnVaLy9yUXRUMys5Z3UxV3ZZMjNrdDcwZEE2TUNHV0VtMWRMQzloWTBmWUxySjlnT0JcbmNUa3piWTdMUms

### Crear secreto estático con instant updates

Creamos un secreto KV v2 en Vault y un recurso `VaultStaticSecret` para sincronizarlo en Kubernetes. El recurso activa `syncConfig.instantUpdates: true`, de modo que VSO se suscribe al event subsystem de Vault y recibe cambios sin esperar al ciclo `refreshAfter`.

In [198]:
%%bash
kubectl create namespace app-1 --dry-run=client -o yaml | kubectl apply -f -

vault kv put secrets/vso-static-demo \
  message="initial-static-value" \
  owner="vso" \
  updated_at="$(date -u +%FT%TZ)"

cat > ${WORKDIR}/static-secret.yaml <<EOF
---
apiVersion: secrets.hashicorp.com/v1beta1
kind: VaultStaticSecret
metadata:
  name: vso-static-secret
  namespace: app-1
spec:
  vaultAuthRef: vault-secrets-operator/example
  namespace: admin
  mount: secrets
  type: kv-v2
  path: vso-static-demo
  refreshAfter: 1h
  destination:
    create: true
    name: vso-static-demo
  syncConfig:
    instantUpdates: true
EOF

kubectl apply -f ${WORKDIR}/static-secret.yaml


namespace/app-1 configured
======== Secret Path ========
secrets/data/vso-static-demo

======= Metadata =======
Key                Value
---                -----
created_time       2026-06-16T16:13:17.865046926Z
custom_metadata    <nil>
deletion_time      n/a
destroyed          false
version            17
vaultstaticsecret.secrets.hashicorp.com/vso-static-secret unchanged


In [200]:
%%bash
# Esperar a que VSO sincronice el secreto por primera vez y mostrar el estado
for i in $(seq 1 30); do
  if kubectl get secret vso-static-demo -n app-1 >/dev/null 2>&1; then
    echo "Secret created by VSO"
    break
  fi
  sleep 2
done

kubectl describe VaultStaticSecret vso-static-secret -n app-1


Secret created by VSO
Name:         vso-static-secret
Namespace:    app-1
Labels:       <none>
Annotations:  <none>
API Version:  secrets.hashicorp.com/v1beta1
Kind:         VaultStaticSecret
Metadata:
  Creation Timestamp:  2026-06-16T16:04:22Z
  Finalizers:
    vaultstaticsecret.secrets.hashicorp.com/finalizer
  Generation:        3
  Resource Version:  3236
  UID:               a4685425-909d-4baa-886e-2b84197546a5
Spec:
  Destination:
    Create:     true
    Name:       vso-static-demo
    Overwrite:  false
    Transformation:
  Hmac Secret Data:  true
  Mount:             secrets
  Namespace:         admin
  Path:              vso-static-demo
  Refresh After:     1h
  Sync Config:
    Instant Updates:  true
  Type:               kv-v2
  Vault Auth Ref:     vault-secrets-operator/example
Status:
  Conditions:
    Last Transition Time:  2026-06-16T16:13:17Z
    Message:               Secret synced, horizon=49m29.07581104s
    Observed Generation:   3
    Reason:                Synce

In [201]:
%%bash
# Decodificar el secreto Kubernetes inicial sincronizado por VSO
kubectl get secret vso-static-demo -n app-1 -o json \
  | jq -r '.data | to_entries[] | "\(.key): \(.value | @base64d)"'


_raw: {"data":{"message":"initial-static-value","owner":"vso","updated_at":"2026-06-16T16:13:17Z"},"metadata":{"created_time":"2026-06-16T16:13:17.865046926Z","custom_metadata":null,"deletion_time":"","destroyed":false,"version":17}}
message: initial-static-value
owner: vso
updated_at: 2026-06-16T16:13:17Z


### Cambiar el secreto en Vault

Actualiza el valor del secreto KV directamente en Vault. Con `instantUpdates: true`, VSO tiene abierta una conexión SSE al event subsystem de Vault: en cuanto se escribe la nueva versión, Vault emite un evento `kv-v2/data-write` que VSO recibe instantáneamente y sincroniza en el secreto Kubernetes sin esperar `refreshAfter`.


In [203]:
! vault kv put secrets/vso-static-demo \
    message="updated-by-vault-event-$(date -u +%Y%m%d%H%M%S)" \
    owner="vso" \
    updated_at="$(date -u +%FT%TZ)"


======== Secret Path ========
secrets/data/vso-static-demo

======= Metadata =======
Key                Value
---                -----
created_time       2026-06-16T16:13:32.552718197Z
custom_metadata    <nil>
deletion_time      n/a
destroyed          false
version            18


In [204]:
# Verificar que VSO recibió el evento y rotó el secreto (buscar SecretRotated en Events)
! kubectl describe VaultStaticSecret vso-static-secret -n app-1


Name:         vso-static-secret
Namespace:    app-1
Labels:       <none>
Annotations:  <none>
API Version:  secrets.hashicorp.com/v1beta1
Kind:         VaultStaticSecret
Metadata:
  Creation Timestamp:  2026-06-16T16:04:22Z
  Finalizers:
    vaultstaticsecret.secrets.hashicorp.com/finalizer
  Generation:        3
  Resource Version:  3258
  UID:               a4685425-909d-4baa-886e-2b84197546a5
Spec:
  Destination:
    Create:     true
    Name:       vso-static-demo
    Overwrite:  false
    Transformation:
  Hmac Secret Data:  true
  Mount:             secrets
  Namespace:         admin
  Path:              vso-static-demo
  Refresh After:     1h
  Sync Config:
    Instant Updates:  true
  Type:               kv-v2
  Vault Auth Ref:     vault-secrets-operator/example
Status:
  Conditions:
    Last Transition Time:  2026-06-16T16:13:32Z
    Message:               Secret synced, horizon=50m39.661546279s
    Observed Generation:   3
    Reason:                Synced
    Status:        

In [205]:
%%bash
# Decodificar el secreto actualizado — el valor de message debe reflejar el nuevo KV
kubectl get secret vso-static-demo -n app-1 -o json \
  | jq -r '.data | to_entries[] | "\(.key): \(.value | @base64d)"'


_raw: {"data":{"message":"updated-by-vault-event-20260616161331","owner":"vso","updated_at":"2026-06-16T16:13:32Z"},"metadata":{"created_time":"2026-06-16T16:13:32.552718197Z","custom_metadata":null,"deletion_time":"","destroyed":false,"version":18}}
message: updated-by-vault-event-20260616161331
owner: vso
updated_at: 2026-06-16T16:13:32Z
